In [1]:
import os
import io
import boto3
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Optional, Dict, Any
from botocore.client import Config
from dotenv import load_dotenv
# Load variables from .env into os.environ
load_dotenv(override=True)

True

In [13]:
from test_s3 import get_s3_client, test_connection, ensure_bucket_exists
import sys
s3_client = get_s3_client()

if not test_connection(s3_client):
    sys.exit(1)

ensure_bucket_exists(s3_client, "01-bronze")

Connected to MinIO. Existing buckets: []
Bucket '01-bronze' created.


True

In [14]:
app = FastAPI(title="Olist Ingestion Webhook Receiver", version="1.0")
BRONZE_BUCKET = os.getenv("BRONZE_BUCKET")


In [ ]:
# ---------------------------------------------------------
# Pydantic Models for Validation
# ---------------------------------------------------------
class WebhookPayload(BaseModel):
    source_table: str  # e.g., "order_payments" or "order_reviews"
    records: List[Dict[str, Any]]

# ---------------------------------------------------------
# API Endpoints
# ---------------------------------------------------------
@app.get("/")
def health_check():
    return {"status": "healthy", "target_bucket": BRONZE_BUCKET}

@app.post("/api/v1/payments/webhook")
def receive_webhook(payload: WebhookPayload):
    """
    Receives streaming operational events, serializes them, 
    and drops them directly into the MinIO Bronze Lake as Parquet/JSON.
    """
    try:
        if not payload.records:
            return {"status": "success", "message": "No records provided."}

        df = pd.DataFrame(payload.records)
        
        # Convert dataframe to JSON/Parquet bytes in memory
        buffer = io.BytesIO()
        df.to_parquet(buffer, index=False)
        buffer.seek(0)
        
        # Generate a unique file name based on table type and timestamp
        file_name = f"{payload.source_table}/batch_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S_%f')}.parquet"
        
        # Upload to MinIO Bronze Layer
        s3_client.upload_fileobj(buffer, BRONZE_BUCKET, file_name)
        
        print(f"📥 [Bronze Landing] Successfully ingested {len(df)} rows into {BRONZE_BUCKET}/{file_name}")
        
        return {
            "status": "success", 
            "inserted_records": len(df), 
            destination_path: f"s3://{BRONZE_BUCKET}/{file_name}"
        }
        
    except Exception as e:
        print(f"❌ Webhook ingestion failed: {str(e)}")
        raise HTTPException(status_code=500, detail=str(e))